# Spaceship Titanic

> Predict which passengers are transported to an alternate dimension

| Submission ke- | Accuracy    | Public Score | Algoritma         |
|----------------|-------------|--------------|-------------------|
| 1              | 0.7768832662449684      | 0.78816      | Cat Boost |
| 2              | 0.7630822311673375      | 0.77437      | K-Nearest Neighbour |
| 3              | 0.7705577918343876      | 0.78489      | Gradient Boosting |
| 4              | 0.7665324899367453      | 0.78115      | Decision Tree |
| 5              | 0.7676825761932144      | 0.78910      | Random Forest Classifier |
| 6              | 0.7757331799884991      |       | XGBoost |

Kaggle competition : [https://www.kaggle.com/competitions/spaceship-titanic](https://www.kaggle.com/competitions/spaceship-titanic)

# 1. Problem Definition

# 2. Data Collection

In [3]:
import pandas as pd

train_df = pd.read_csv('dataset/train.csv')
test_df = pd.read_csv('dataset/test.csv')
submission_df = pd.read_csv('dataset/sample_submission.csv')

# 3. Exploratory Data Analysis

In [ ]:
# Daftar kolom numerik
numerical_cols = train_df.select_dtypes(include=['number']).columns.tolist()
print("Kolom numerik:", numerical_cols)

# Daftar kolom object (kategori atau string)
object_cols = train_df.select_dtypes(include=['object']).columns.tolist()
print("Kolom object:", object_cols)

del numerical_cols, object_cols

Kolom numerik: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
Kolom object: ['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP', 'Name']


In [14]:
for col_name in train_df.columns:
    dtype = train_df[col_name].dtype
    if dtype == 'float64' or dtype == 'bool':
        continue
    unique_count = len(train_df[col_name].unique())
    print(f"{col_name.ljust(13)} dtype: {str(dtype).ljust(8)} unique values: {unique_count}")

del col_name, unique_count, dtype

PassengerId   dtype: object   unique values: 8693
HomePlanet    dtype: object   unique values: 4
CryoSleep     dtype: object   unique values: 3
Cabin         dtype: object   unique values: 6561
Destination   dtype: object   unique values: 4
VIP           dtype: object   unique values: 3
Name          dtype: object   unique values: 8474


In [12]:
for col_name in test_df.columns:
    dtype = test_df[col_name].dtype
    if dtype == 'float64':
        continue
    unique_count = len(test_df[col_name].unique())
    print(f"{col_name.ljust(13)} dtype: {str(dtype).ljust(8)} unique values: {unique_count}")

del col_name, unique_count, dtype

PassengerId   dtype: object   unique values: 4277
HomePlanet    dtype: object   unique values: 4
CryoSleep     dtype: object   unique values: 3
Cabin         dtype: object   unique values: 3266
Destination   dtype: object   unique values: 4
VIP           dtype: object   unique values: 3
Name          dtype: object   unique values: 4177


In [20]:
# Cek kolom bertipe object
print("Missing values (dtype: object):")
print(train_df.loc[:, train_df.dtypes == 'object'].isna().sum())

print("\nMissing values (dtype: float64):")
print(train_df.loc[:, train_df.dtypes == 'float64'].isna().sum())

print("\nMissing values (dtype: bool):")
print(train_df.loc[:, train_df.dtypes == 'bool'].isna().sum())

Missing values (dtype: object):
PassengerId      0
HomePlanet     201
CryoSleep      217
Cabin          199
Destination    182
VIP            203
Name           200
dtype: int64

Missing values (dtype: float64):
Age             179
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
dtype: int64

Missing values (dtype: bool):
Transported    0
dtype: int64


In [47]:
print("Train dataset:")
for col_name in train_df.columns:
    dtype = train_df[col_name].dtype
    if dtype == 'float64':
        mean = train_df[col_name].mean()
        median = train_df[col_name].median()
        print(f"{col_name.ljust(15)} mean: {mean:10.4f}  median: {median:10.4f}")
    else:
        continue

del mean, median, col_name, dtype

Train dataset:
Age             mean:    28.8279  median:    27.0000
RoomService     mean:   224.6876  median:     0.0000
FoodCourt       mean:   458.0772  median:     0.0000
ShoppingMall    mean:   173.7292  median:     0.0000
Spa             mean:   311.1388  median:     0.0000
VRDeck          mean:   304.8548  median:     0.0000


In [48]:
print("Test dataset:")
for col_name in test_df.columns:
    dtype = test_df[col_name].dtype
    if dtype == 'float64':
        mean = test_df[col_name].mean()
        median = test_df[col_name].median()
        print(f"{col_name.ljust(15)} mean: {mean:10.4f}  median: {median:10.4f}")
    else:
        continue

del mean, median, col_name, dtype

Test dataset:
Age             mean:    28.6581  median:    26.0000
RoomService     mean:   219.2663  median:     0.0000
FoodCourt       mean:   439.4843  median:     0.0000
ShoppingMall    mean:   177.2955  median:     0.0000
Spa             mean:   303.0524  median:     0.0000
VRDeck          mean:   310.7100  median:     0.0000


In [45]:
missing_rows = train_df.isna().any(axis=1).sum()
print(f"Jumlah baris yang punya missing value: {missing_rows}")

del missing_rows

Jumlah baris yang punya missing value: 2087


In [ ]:
train_df['Age'] = train_df['Age'].fillna(27)
test_df['Age'] = test_df['Age'].fillna(27)

In [50]:
missing_rows = train_df.isna().any(axis=1).sum()
print(f"Jumlah baris yang punya missing value: {missing_rows}")

del missing_rows

Jumlah baris yang punya missing value: 1936


In [ ]:
for col_name in numerical_cols:
    median = train_df[col_name].median()
    train_df[col_name] = train_df[col_name].fillna(median)

for col_name in numerical_cols:
    median = test_df[col_name].median()
    test_df[col_name] = test_df[col_name].fillna(median)
        
del col_name, median

In [56]:
missing_rows = train_df.isna().any(axis=1).sum()
print(f"Jumlah baris yang punya missing value: {missing_rows}")

del missing_rows

Jumlah baris yang punya missing value: 1134


In [70]:
print("Unique value list on Train dataset:")
for col_name in  ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']:
    unique_values = train_df[col_name].unique().tolist()
    print(f'{col_name.ljust(15)}: {unique_values}')
    
del unique_values, col_name

Unique value list on Train dataset:
HomePlanet     : ['Europa', 'Earth', 'Mars', nan]
CryoSleep      : [False, True, nan]
Destination    : ['TRAPPIST-1e', 'PSO J318.5-22', '55 Cancri e', nan]
VIP            : [False, True, nan]


In [71]:
print("Unique value list on Test dataset:")
for col_name in  ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']:
    unique_values = test_df[col_name].unique().tolist()
    print(f'{col_name.ljust(15)}: {unique_values}')
    
del unique_values, col_name

Unique value list on Test dataset:
HomePlanet     : ['Earth', 'Europa', 'Mars', nan]
CryoSleep      : [True, False, nan]
Destination    : ['TRAPPIST-1e', '55 Cancri e', 'PSO J318.5-22', nan]
VIP            : [False, nan, True]


In [68]:
print("Most frequent on Train dataset:")
for col_name in  ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']:
    most_frequent = train_df[col_name].mode()[0]
    print(f'{col_name.ljust(15)}: {most_frequent}')
    
del most_frequent, col_name

Most frequent on Train dataset:
HomePlanet     : Earth
CryoSleep      : False
Destination    : TRAPPIST-1e
VIP            : False


In [69]:
print("Most frequent on Test dataset:")
for col_name in  ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']:
    most_frequent = test_df[col_name].mode()[0]
    print(f'{col_name.ljust(15)}: {most_frequent}')
    
del most_frequent, col_name

Most frequent on Test dataset:
HomePlanet     : Earth
CryoSleep      : False
Destination    : TRAPPIST-1e
VIP            : False


In [76]:
for col_name in  ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']:
    most_frequent = train_df[col_name].mode()[0]
    train_df[col_name] = train_df[col_name].fillna(most_frequent)

del most_frequent, col_name

In [77]:
for col_name in  ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']:
    most_frequent = test_df[col_name].mode()[0]
    test_df[col_name] = test_df[col_name].fillna(most_frequent)

del most_frequent, col_name

In [79]:
train_df['CryoSleep'] = train_df['CryoSleep'].replace({True: 1, False: 0})
train_df['VIP'] = train_df['VIP'].replace({True: 1, False: 0})


In [80]:
test_df['CryoSleep'] = test_df['CryoSleep'].replace({True: 1, False: 0})
test_df['VIP'] = test_df['VIP'].replace({True: 1, False: 0})

In [ ]:
for col_name in ['Cabin', 'Name']:
    train_df[col_name] = train_df[col_name].fillna('unknown')
    test_df[col_name] = test_df[col_name].fillna('unknown')
    
del col_name

In [87]:
from scipy.stats import chi2_contingency

# Fungsi untuk melakukan Chi-Square Test
def chi_square_test(df, feature, target):
    contingency_table = pd.crosstab(df[feature], df[target])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    return p

features = ['HomePlanet', 'Cabin', 'Destination', 'Name']
# Menghitung nilai p untuk setiap fitur
p_values = {feature: chi_square_test(train_df, feature, 'Transported') for feature in features}

# Membuat DataFrame untuk menampilkan hasil
p_values_df = pd.DataFrame(list(p_values.items()), columns=['Feature', 'P-value'])
p_values_df = p_values_df.sort_values(by='P-value')

print(p_values_df)

# del features
del p_values, p_values_df, features

       Feature       P-value
0   HomePlanet  5.549632e-70
2  Destination  1.194789e-23
1        Cabin  6.198146e-03
3         Name  4.918026e-01


In [88]:
from sklearn.feature_selection import f_classif, SelectKBest

num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
X_anova = train_df[num_cols]
y_anova = train_df['Transported']

# Menggunakan ANOVA F-test untuk feature selection
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X_anova, y_anova)

# Mendapatkan skor ANOVA F untuk setiap fitur
scores = selector.scores_

# Membuat DataFrame untuk menampilkan hasil
num_feature_scores = pd.DataFrame({'Feature': X_anova.columns, 'ANOVA F-Score': scores})
num_feature_scores = num_feature_scores.sort_values(by='ANOVA F-Score', ascending=False)

print(num_feature_scores)

del num_cols, X_anova, y_anova, selector, scores, num_feature_scores

        Feature  ANOVA F-Score
1   RoomService     536.491728
4           Spa     435.917591
5        VRDeck     380.771546
0           Age      48.156922
2     FoodCourt      18.096177
3  ShoppingMall       0.766539


In [107]:
features_selected = ['RoomService', 'Spa', 'VRDeck']
# 'HomePlanet', 'Destination'

In [95]:
train_df['HomePlanet'].unique()

array(['Europa', 'Earth', 'Mars'], dtype=object)

In [96]:
test_df['HomePlanet'].unique()

array(['Earth', 'Europa', 'Mars'], dtype=object)

In [94]:
train_df['Destination'].unique()

array(['TRAPPIST-1e', 'PSO J318.5-22', '55 Cancri e'], dtype=object)

In [97]:
test_df['Destination'].unique()

array(['TRAPPIST-1e', '55 Cancri e', 'PSO J318.5-22'], dtype=object)

In [99]:
train_HomePlanet = train_df['HomePlanet']
train_HomePlanet = pd.get_dummies(train_HomePlanet, prefix='HomePlanet')
train_HomePlanet

,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars
0,False,True,False
1,True,False,False
2,False,True,False
3,False,True,False
4,True,False,False
...,...,...,...
8688,False,True,False
8689,True,False,False
8690,True,False,False
8691,False,True,False


In [100]:
test_HomePlanet = test_df['HomePlanet']
test_HomePlanet = pd.get_dummies(test_HomePlanet, prefix='HomePlanet')
test_HomePlanet

,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars
0,True,False,False
1,True,False,False
2,False,True,False
3,False,True,False
4,True,False,False
...,...,...,...
4272,True,False,False
4273,True,False,False
4274,False,False,True
4275,False,True,False


In [101]:
train_Destination = train_df['Destination']
train_Destination = pd.get_dummies(train_Destination, prefix='Destination')
train_Destination

,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
0,False,False,True
1,False,False,True
2,False,False,True
3,False,False,True
4,False,False,True
...,...,...,...
8688,True,False,False
8689,False,True,False
8690,False,False,True
8691,True,False,False


In [102]:
test_Destination = test_df['Destination']
test_Destination = pd.get_dummies(test_Destination, prefix='Destination')
test_Destination

,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
0,False,False,True
1,False,False,True
2,True,False,False
3,False,False,True
4,False,False,True
...,...,...,...
4272,False,False,True
4273,False,False,True
4274,True,False,False
4275,False,False,True


In [ ]:
# Mengclone train dataframe dengan fitur terpilih ke dataframe baru
X = train_df[features_selected]
y = train_df['Transported']

X_pred = test_df[features_selected]

X = pd.concat([X, train_HomePlanet, train_Destination], axis=1)
X_pred = pd.concat([X_pred, test_HomePlanet, test_Destination], axis=1)

del train_Destination, train_HomePlanet, test_Destination, test_HomePlanet
del features_selected

In [122]:
# Public score : 0.78816
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# Membuat model CatBoost
model = CatBoostClassifier(iterations=100, learning_rate=0.1, depth=6, verbose=0)

# Membagi data menjadi training dan testing set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Melatih model
model.fit(X_train, y_train)

# Memprediksi pada data testing
y_pred = model.predict(X_test)

# Evaluasi kinerja model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("CatBoost")
print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)

# Cross-validation untuk evaluasi lebih lanjut
cv_scores = cross_val_score(model, X, y, cv=2)
print(f"Cross-Validation Accuracy: {cv_scores.mean()} ± {cv_scores.std()}")

del y_pred, accuracy, report, cv_scores

CatBoost
Accuracy: 0.7768832662449684
Classification Report:
              precision    recall  f1-score   support

       False       0.82      0.70      0.76       861
        True       0.74      0.85      0.79       878

    accuracy                           0.78      1739
   macro avg       0.78      0.78      0.78      1739
weighted avg       0.78      0.78      0.78      1739

Cross-Validation Accuracy: 0.7835049715589543 ± 0.008717761989135986


In [ ]:
# Public score 0.78115
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Membuat model Decision Tree
model = DecisionTreeClassifier(max_depth=6, random_state=42)

# Melatih model
model.fit(X_train, y_train)

# Memprediksi pada data testing
y_pred = model.predict(X_test)

# Evaluasi kinerja model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("Decision Tree")
print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)

# Cross-validation untuk evaluasi lebih lanjut
cv_scores = cross_val_score(model, X, y, cv=2)
print(f"Cross-Validation Accuracy: {cv_scores.mean()} ± {cv_scores.std()}")

# Membersihkan variabel
del y_pred, accuracy, report, cv_scores

# submission_df['Transported'] = model.predict(X_pred)
# submission_df.to_csv(f'submission-{4}.csv', index=False, header=True)


Decision Tree
Accuracy: 0.7665324899367453
Classification Report:
              precision    recall  f1-score   support

       False       0.82      0.68      0.74       861
        True       0.73      0.85      0.79       878

    accuracy                           0.77      1739
   macro avg       0.77      0.77      0.76      1739
weighted avg       0.77      0.77      0.76      1739

Cross-Validation Accuracy: 0.7714262477012832 ± 0.008371267254998427


In [ ]:
# Public score 0.78910
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Membuat model Random Forest
model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)

# Melatih model
model.fit(X_train, y_train)

# Memprediksi pada data testing
y_pred = model.predict(X_test)

# Evaluasi kinerja model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("Random Forest Classifier")
print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)

# Cross-validation untuk evaluasi lebih lanjut
cv_scores = cross_val_score(model, X, y, cv=2)
print(f"Cross-Validation Accuracy: {cv_scores.mean()} ± {cv_scores.std()}")

# Membersihkan variabel
del y_pred, accuracy, report, cv_scores

submission_df['Transported'] = model.predict(X_pred)
submission_df.to_csv(f'submission-{5}.csv', index=False, header=True)


Random Forest Classifier
Accuracy: 0.7676825761932144
Classification Report:
              precision    recall  f1-score   support

       False       0.79      0.73      0.76       861
        True       0.75      0.81      0.78       878

    accuracy                           0.77      1739
   macro avg       0.77      0.77      0.77      1739
weighted avg       0.77      0.77      0.77      1739

Cross-Validation Accuracy: 0.7797084563876616 ± 0.005611377942757123


In [ ]:
# Public score 0.78396
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Membuat model XGBoost
model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, 
                      use_label_encoder=False, eval_metric='mlogloss', verbosity=0, random_state=42)

# Melatih model
model.fit(X_train, y_train)

# Memprediksi pada data testing
y_pred = model.predict(X_test)

# Evaluasi kinerja model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("XGBoost")
print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)

# Cross-validation untuk evaluasi lebih lanjut
cv_scores = cross_val_score(model, X, y, cv=2)
print(f"Cross-Validation Accuracy: {cv_scores.mean()} ± {cv_scores.std()}")

# Membersihkan variabel
del y_pred, accuracy, report, cv_scores

# submission_df['Transported'] = model.predict(X_pred)
# submission_df.to_csv(f'submission-{6}.csv', index=False, header=True)


XGBoost
Accuracy: 0.7757331799884991
Classification Report:
              precision    recall  f1-score   support

       False       0.83      0.69      0.75       861
        True       0.74      0.86      0.79       878

    accuracy                           0.78      1739
   macro avg       0.78      0.77      0.77      1739
weighted avg       0.78      0.78      0.77      1739

Cross-Validation Accuracy: 0.7798238223016629 ± 0.008487268356413391


In [ ]:
# Public score 0.78489
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Membuat model Gradient Boosting
model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)

# Melatih model
model.fit(X_train, y_train)

# Memprediksi pada data testing
y_pred = model.predict(X_test)

# Evaluasi kinerja model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("Gradient Boosting")
print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)

# Cross-validation untuk evaluasi lebih lanjut
cv_scores = cross_val_score(model, X, y, cv=2)
print(f"Cross-Validation Accuracy: {cv_scores.mean()} ± {cv_scores.std()}")

# Membersihkan variabel
del y_pred, accuracy, report, cv_scores


Gradient Boosting
Accuracy: 0.7705577918343876
Classification Report:
              precision    recall  f1-score   support

       False       0.82      0.69      0.75       861
        True       0.74      0.85      0.79       878

    accuracy                           0.77      1739
   macro avg       0.78      0.77      0.77      1739
weighted avg       0.78      0.77      0.77      1739

Cross-Validation Accuracy: 0.7747621990654063 ± 0.007796475577943773


In [134]:
# Public score 0.77437
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# for k in range (1, 20): 
# 	knn = KNeighborsClassifier(n_neighbors=k)
# 	knn.fit(X_train, y_train)
# 	y_pred = knn.predict(X_test)
# 	accuracy = accuracy_score(y_test, y_pred)
# 	print(f"Akurasi model KNN saat k={k}: {accuracy * 100:.2f}%")

knn = KNeighborsClassifier(n_neighbors=12)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print('K-Nearest Neighbour')
print(f"Akurasi model K-Nearest Neighbour: {accuracy}")

K-Nearest Neighbour
Akurasi model K-Nearest Neighbour: 0.7630822311673375


In [136]:
submission_df['Transported'] = model.predict(X_pred)

In [ ]:
nomor = 10
submission_df.to_csv(f'submission-{nomor}.csv', index=False, header=True)

del nomor